# PySpark Tutorial - Quick Start Guide

Welcome to this comprehensive PySpark tutorial! This notebook will help you learn PySpark fundamentals with practical examples.

## What you'll learn:
1. Setting up PySpark
2. Loading and exploring data
3. DataFrame operations
4. Filtering and selecting data
5. Aggregations and GroupBy
6. Joins
7. SQL queries
8. Writing data to files
9. Window functions
10. Advanced transformations

## 1. Install and Import PySpark

First, let's install PySpark (if not already installed) and import the necessary modules.

In [164]:
# Install PySpark (run this if not already installed)
# !pip install pyspark

# Import necessary modules
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum, avg, max, min, when, lit, upper, lower, concat, substring
from pyspark.sql.functions import year, month, datediff, current_date, to_date, desc, asc
from pyspark.sql.window import Window
from pyspark.sql import functions as F

print("PySpark modules imported successfully!")

PySpark modules imported successfully!


## 2. Initialize SparkSession

SparkSession is the entry point for working with PySpark DataFrames. Let's create one.

In [165]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = f"/opt/homebrew/opt/openjdk@17/bin:{os.environ['PATH']}"

# Create SparkSession
spark = SparkSession.builder \
    .appName("PySpark Tutorial") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print("SparkSession created successfully!")

Spark Version: 4.0.2
SparkSession created successfully!


## 3. Load Data into DataFrame

We'll load the sample employee data CSV file that was created for this tutorial.

In [166]:
# Load CSV file into DataFrame
df = spark.read.csv("sample_employees.csv", header=True, inferSchema=True)

print("Data loaded successfully!")
print(f"Number of rows: {df.count()}")
print(f"Number of columns: {len(df.columns)}")

Data loaded successfully!
Number of rows: 20
Number of columns: 7


## 4. Basic DataFrame Operations

Let's explore basic operations to understand our data better.

In [167]:
# Display the first few rows
print("First 5 rows:")
df.show(5)
df.columns

First 5 rows:
+-----------+-------------+-----------+------+---+----------+-------------+
|employee_id|         name| department|salary|age| join_date|     location|
+-----------+-------------+-----------+------+---+----------+-------------+
|        101|   John Smith|Engineering| 95000| 32|2019-03-15|San Francisco|
|        102|Sarah Johnson|  Marketing| 75000| 28|2020-06-01|     New York|
|        103|Michael Brown|Engineering|105000| 35|2018-01-20|San Francisco|
|        104|  Emily Davis|      Sales| 68000| 29|2021-02-10|      Chicago|
|        105| David Wilson|Engineering| 88000| 31|2019-11-05|      Seattle|
+-----------+-------------+-----------+------+---+----------+-------------+
only showing top 5 rows


['employee_id', 'name', 'department', 'salary', 'age', 'join_date', 'location']

In [168]:
# Display schema
print("Schema of the DataFrame:")
df.printSchema()

Schema of the DataFrame:
root
 |-- employee_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- join_date: date (nullable = true)
 |-- location: string (nullable = true)



In [169]:
# Get column names and data types
print("Column names:", df.columns)
print("\nData types:")
df.dtypes

Column names: ['employee_id', 'name', 'department', 'salary', 'age', 'join_date', 'location']

Data types:


[('employee_id', 'int'),
 ('name', 'string'),
 ('department', 'string'),
 ('salary', 'int'),
 ('age', 'int'),
 ('join_date', 'date'),
 ('location', 'string')]

In [170]:
# Get statistical summary
print("Statistical summary of numerical columns:")
df.describe().show()

Statistical summary of numerical columns:
+-------+-----------------+-------------+-----------+-----------------+-----------------+--------+
|summary|      employee_id|         name| department|           salary|              age|location|
+-------+-----------------+-------------+-----------+-----------------+-----------------+--------+
|  count|               20|           20|         20|               20|               20|      20|
|   mean|            110.5|         NULL|       NULL|          84200.0|             31.2|    NULL|
| stddev|5.916079783099616|         NULL|       NULL|11794.73566781741|3.833371471968933|    NULL|
|    min|              101| Amanda Lewis|Engineering|            68000|               25|  Austin|
|    max|              120|Sarah Johnson|      Sales|           105000|               38| Seattle|
+-------+-----------------+-------------+-----------+-----------------+-----------------+--------+



## 5. Filtering and Selecting Data

Learn how to select specific columns and filter rows based on conditions.

In [171]:
# Select specific columns
print("Select name and salary columns:")
# Select One column
print(df.select("name").show(5))
df.select("name", "salary").show(5)

# Select name and salary coumns where salary is greater than 60000
print("Select name and salary where salary > 60000:")
df_new = df.filter(col("salary") > 60000)
df_new.select("name", "salary").show(5)

Select name and salary columns:
+-------------+
|         name|
+-------------+
|   John Smith|
|Sarah Johnson|
|Michael Brown|
|  Emily Davis|
| David Wilson|
+-------------+
only showing top 5 rows
None
+-------------+------+
|         name|salary|
+-------------+------+
|   John Smith| 95000|
|Sarah Johnson| 75000|
|Michael Brown|105000|
|  Emily Davis| 68000|
| David Wilson| 88000|
+-------------+------+
only showing top 5 rows
Select name and salary where salary > 60000:
+-------------+------+
|         name|salary|
+-------------+------+
|   John Smith| 95000|
|Sarah Johnson| 75000|
|Michael Brown|105000|
|  Emily Davis| 68000|
| David Wilson| 88000|
+-------------+------+
only showing top 5 rows


In [172]:
# Filter rows - employees with salary > 90000
print("Employees with salary > 90000:")
df.filter(col("salary") > 90000).show()



Employees with salary > 90000:
+-----------+----------------+-----------+------+---+----------+-------------+
|employee_id|            name| department|salary|age| join_date|     location|
+-----------+----------------+-----------+------+---+----------+-------------+
|        101|      John Smith|Engineering| 95000| 32|2019-03-15|San Francisco|
|        103|   Michael Brown|Engineering|105000| 35|2018-01-20|San Francisco|
|        109|    James Thomas|Engineering| 98000| 34|2018-09-08|San Francisco|
|        110|Patricia Jackson|    Finance| 91000| 36|2017-12-01|     New York|
|        112| Jennifer Harris|Engineering|102000| 38|2016-04-25|      Seattle|
|        115| Daniel Robinson|    Finance| 94000| 37|2017-06-14|     New York|
|        117|     Joseph Hall|Engineering| 99000| 33|2018-11-20|San Francisco|
+-----------+----------------+-----------+------+---+----------+-------------+



In [173]:
# Multiple conditions - Engineering department with salary > 90000
print("Engineering employees with salary > 90000:")
df.filter((col("department") == "Engineering") & (col("salary") > 90000)).show()

Engineering employees with salary > 90000:
+-----------+---------------+-----------+------+---+----------+-------------+
|employee_id|           name| department|salary|age| join_date|     location|
+-----------+---------------+-----------+------+---+----------+-------------+
|        101|     John Smith|Engineering| 95000| 32|2019-03-15|San Francisco|
|        103|  Michael Brown|Engineering|105000| 35|2018-01-20|San Francisco|
|        109|   James Thomas|Engineering| 98000| 34|2018-09-08|San Francisco|
|        112|Jennifer Harris|Engineering|102000| 38|2016-04-25|      Seattle|
|        117|    Joseph Hall|Engineering| 99000| 33|2018-11-20|San Francisco|
+-----------+---------------+-----------+------+---+----------+-------------+



In [174]:
# Using where() - same as filter()
print("Employees in New York or San Francisco:")
df.where(col("location").isin(["New York", "San Francisco"])).show()

Employees in New York or San Francisco:
+-----------+----------------+-----------+------+---+----------+-------------+
|employee_id|            name| department|salary|age| join_date|     location|
+-----------+----------------+-----------+------+---+----------+-------------+
|        101|      John Smith|Engineering| 95000| 32|2019-03-15|San Francisco|
|        102|   Sarah Johnson|  Marketing| 75000| 28|2020-06-01|     New York|
|        103|   Michael Brown|Engineering|105000| 35|2018-01-20|San Francisco|
|        106|Jessica Martinez|         HR| 72000| 27|2020-08-22|     New York|
|        109|    James Thomas|Engineering| 98000| 34|2018-09-08|San Francisco|
|        110|Patricia Jackson|    Finance| 91000| 36|2017-12-01|     New York|
|        115| Daniel Robinson|    Finance| 94000| 37|2017-06-14|     New York|
|        117|     Joseph Hall|Engineering| 99000| 33|2018-11-20|San Francisco|
|        119|      Ryan Young|    Finance| 89000| 35|2018-02-28|     New York|
+-----------

## 6. Adding and Modifying Columns

Learn how to add new columns or modify existing ones using `withColumn()`.

In [175]:
# Add a new column - annual bonus (10% of salary)
df_with_bonus = df.withColumn("bonus", col("salary") * 0.10)
df_with_bonus.select("name", "salary", "bonus").show(5)

+-------------+------+-------+
|         name|salary|  bonus|
+-------------+------+-------+
|   John Smith| 95000| 9500.0|
|Sarah Johnson| 75000| 7500.0|
|Michael Brown|105000|10500.0|
|  Emily Davis| 68000| 6800.0|
| David Wilson| 88000| 8800.0|
+-------------+------+-------+
only showing top 5 rows


In [176]:
# Add conditional column - salary category
df_with_category = df.withColumn(
    "salary_category",
    when(col("salary") >= 95000, "High")
    .when(col("salary") >= 75000, "Medium")
    .otherwise("Low")
)
df_with_category.select("name", "salary", "salary_category").show(5)

+-------------+------+---------------+
|         name|salary|salary_category|
+-------------+------+---------------+
|   John Smith| 95000|           High|
|Sarah Johnson| 75000|         Medium|
|Michael Brown|105000|           High|
|  Emily Davis| 68000|            Low|
| David Wilson| 88000|         Medium|
+-------------+------+---------------+
only showing top 5 rows


## 7. GroupBy and Aggregations

Aggregation functions let you summarise data across groups or the whole DataFrame.

| Function | Description |
|---|---|
| `count()` | Number of rows |
| `sum()` | Sum of a numeric column |
| `avg()` / `mean()` | Average value |
| `min()` / `max()` | Minimum / Maximum value |
| `countDistinct()` | Count of unique values |
| `stddev()` | Standard deviation |
| `variance()` | Variance |
| `first()` / `last()` | First / Last value in a group |
| `collect_list()` | All values as a list (with duplicates) |
| `collect_set()` | All unique values as a set |
| `percentile_approx()` | Approximate percentile |
| `sumDistinct()` | Sum of distinct values |

In [177]:
# ── 1. count() ──
# Count total rows and rows per group
print("Total employees:", df.count())

print("\nEmployee count per department:")
df.groupBy("department").count().show()
df.groupBy(col("department")).avg("salary").show()
df.groupBy("department").mean("salary").show()

Total employees: 20

Employee count per department:
+-----------+-----+
| department|count|
+-----------+-----+
|      Sales|    4|
|Engineering|    6|
|         HR|    3|
|    Finance|    3|
|  Marketing|    4|
+-----------+-----+

+-----------+-----------------+
| department|      avg(salary)|
+-----------+-----------------+
|      Sales|          74500.0|
|Engineering|97833.33333333333|
|         HR|71333.33333333333|
|    Finance|91333.33333333333|
|  Marketing|          77750.0|
+-----------+-----------------+

+-----------+-----------------+
| department|      avg(salary)|
+-----------+-----------------+
|      Sales|          74500.0|
|Engineering|97833.33333333333|
|         HR|71333.33333333333|
|    Finance|91333.33333333333|
|  Marketing|          77750.0|
+-----------+-----------------+



In [178]:
# ── 2. sum() and avg() ──
from pyspark.sql.functions import sum, avg

print("Total salary paid per department:")
df.groupBy("department").agg(avg("salary")).show()
df.groupBy("department").avg("salary").show()
df.groupBy("department").agg(sum("salary").alias("total_salary")).orderBy("total_salary", ascending=False).show()
print(df.columns)

#print("Average salary per department:")
#df.groupBy("department").agg(avg("salary").alias("avg_salary")).orderBy("avg_salary", ascending=False).show()

Total salary paid per department:
+-----------+-----------------+
| department|      avg(salary)|
+-----------+-----------------+
|      Sales|          74500.0|
|Engineering|97833.33333333333|
|         HR|71333.33333333333|
|    Finance|91333.33333333333|
|  Marketing|          77750.0|
+-----------+-----------------+

+-----------+-----------------+
| department|      avg(salary)|
+-----------+-----------------+
|      Sales|          74500.0|
|Engineering|97833.33333333333|
|         HR|71333.33333333333|
|    Finance|91333.33333333333|
|  Marketing|          77750.0|
+-----------+-----------------+

+-----------+------------+
| department|total_salary|
+-----------+------------+
|Engineering|      587000|
|  Marketing|      311000|
|      Sales|      298000|
|    Finance|      274000|
|         HR|      214000|
+-----------+------------+

['employee_id', 'name', 'department', 'salary', 'age', 'join_date', 'location']


In [424]:
# ── 3. min(), max(), and multiple aggregations at once ──
from pyspark.sql.functions import count, min, max

print("Full salary stats per location, min salary, max salary and average salary:")
df.groupBy("location").agg(avg("salary").alias("avg_salary"), min("salary").alias("min_salary"), max("salary").alias("max_salary")).show()

df.groupBy("location").agg(
    count("*").alias("employee_count"),
    avg("salary").alias("avg_salary"),
    min("salary").alias("min_salary"),
    max("salary").alias("max_salary"),
    (max("salary") - min("salary")).alias("salary_range")
).orderBy("avg_salary", ascending=False).show()

Full salary stats per location, min salary, max salary and average salary:
+-------------+-----------------+----------+----------+
|     location|       avg_salary|min_salary|max_salary|
+-------------+-----------------+----------+----------+
|  Los Angeles|          81000.0|     81000|     81000|
|San Francisco|          99250.0|     95000|    105000|
|       Austin|          71000.0|     69000|     73000|
|      Chicago|75666.66666666667|     68000|     82000|
|      Seattle|          95000.0|     88000|    102000|
|        Miami|          71000.0|     71000|     71000|
|     New York|          84200.0|     72000|     94000|
|       Boston|          77500.0|     76000|     79000|
+-------------+-----------------+----------+----------+

+-------------+--------------+-----------------+----------+----------+------------+
|     location|employee_count|       avg_salary|min_salary|max_salary|salary_range|
+-------------+--------------+-----------------+----------+----------+------------+


In [425]:
# ── 4. countDistinct() and sumDistinct() ──
from pyspark.sql.functions import countDistinct, sumDistinct

print("Number of distinct locations:")
df.agg(countDistinct("location").alias("distinct_locations")).show()

print("Distinct departments per location (count):")
df.groupBy("location").agg(
    countDistinct("department").alias("distinct_depts")
).show()

Number of distinct locations:
+------------------+
|distinct_locations|
+------------------+
|                 8|
+------------------+

Distinct departments per location (count):
+-------------+--------------+
|     location|distinct_depts|
+-------------+--------------+
|  Los Angeles|             1|
|San Francisco|             1|
|       Austin|             1|
|      Chicago|             1|
|      Seattle|             1|
|        Miami|             1|
|     New York|             3|
|       Boston|             1|
+-------------+--------------+



In [426]:
# ── 5. stddev() and variance() ──
from pyspark.sql.functions import stddev, variance

print("Salary spread per department (stddev + variance):")
df.groupBy("department").agg(
    avg("salary").alias("avg_salary"),
    stddev("salary").alias("stddev_salary"),
    variance("salary").alias("variance_salary")
).orderBy("stddev_salary", ascending=False).show()

Salary spread per department (stddev + variance):
+-----------+-----------------+------------------+-------------------+
| department|       avg_salary|     stddev_salary|    variance_salary|
+-----------+-----------------+------------------+-------------------+
|      Sales|          74500.0| 6244.997998398398|3.899999999999999E7|
|Engineering|97833.33333333333| 5913.261931173578|3.496666666666667E7|
|  Marketing|          77750.0| 2753.785273643049|  7583333.333333324|
|    Finance|91333.33333333333|2516.6114784235833|  6333333.333333333|
|         HR|71333.33333333333|2081.6659994661327|  4333333.333333333|
+-----------+-----------------+------------------+-------------------+



In [398]:
# ── 6. first() and last() ──
# Returns the first/last value encountered in each group (order is not guaranteed unless sorted first)
from pyspark.sql.functions import first, last

print("First and last employee name alphabetically per department:")
df.orderBy("name").groupBy("department").agg(
    first("name").alias("first_name"),
    last("name").alias("last_name")
).show()

First and last employee name alphabetically per department:
+-----------+-----------------+-------------+
| department|       first_name|    last_name|
+-----------+-----------------+-------------+
|Engineering|     David Wilson|Michael Brown|
|    Finance|  Daniel Robinson|   Ryan Young|
|         HR|     Amanda Lewis|   Karen King|
|  Marketing|  Elizabeth Allen|Sarah Johnson|
|      Sales|Christopher White|Robert Taylor|
+-----------+-----------------+-------------+



In [399]:
# ── 7. collect_list() and collect_set() ──
# collect_list → all values including duplicates
# collect_set  → only unique values (unordered)
from pyspark.sql.functions import collect_list, collect_set, concat_ws

print("All employee names per department (as a list):")
df.groupBy("department").agg(
    concat_ws(", ", collect_list("name")).alias("employees")
).show(truncate=False)

print("Unique locations per department:")
df.groupBy("department").agg(
    collect_set("location").alias("unique_locations")
).show(truncate=False)

All employee names per department (as a list):
+-----------+-----------------------------------------------------------------------------------+
|department |employees                                                                          |
+-----------+-----------------------------------------------------------------------------------+
|Sales      |Emily Davis, Robert Taylor, Christopher White, Michelle Walker                     |
|Engineering|John Smith, Michael Brown, David Wilson, James Thomas, Jennifer Harris, Joseph Hall|
|HR         |Jessica Martinez, Amanda Lewis, Karen King                                         |
|Finance    |Patricia Jackson, Daniel Robinson, Ryan Young                                      |
|Marketing  |Sarah Johnson, Linda Anderson, Matthew Clark, Elizabeth Allen                      |
+-----------+-----------------------------------------------------------------------------------+

Unique locations per department:
+-----------+------------------------

In [400]:
# ── 8. percentile_approx() ──
# Compute approximate percentiles (25th, 50th/median, 75th)
from pyspark.sql.functions import percentile_approx

print("Salary percentiles (25th / 50th / 75th) per department:")
df.groupBy("department").agg(
    percentile_approx("salary", 0.25).alias("p25"),
    percentile_approx("salary", 0.50).alias("median"),
    percentile_approx("salary", 0.75).alias("p75")
).orderBy("median", ascending=False).show()

Salary percentiles (25th / 50th / 75th) per department:
+-----------+-----+------+------+
| department|  p25|median|   p75|
+-----------+-----+------+------+
|Engineering|95000| 98000|102000|
|    Finance|89000| 91000| 94000|
|  Marketing|75000| 76000| 79000|
|         HR|69000| 72000| 73000|
|      Sales|68000| 71000| 77000|
+-----------+-----+------+------+



In [401]:
# ── 9. Global aggregations (no groupBy) ──
# Use .agg() directly on the DataFrame to get company-wide stats
print("Company-wide salary statistics:")
df.agg(
    count("*").alias("total_employees"),
    sum("salary").alias("total_payroll"),
    avg("salary").alias("avg_salary"),
    min("salary").alias("min_salary"),
    max("salary").alias("max_salary"),
    stddev("salary").alias("stddev_salary")
).show()

Company-wide salary statistics:
+---------------+-------------+----------+----------+----------+-----------------+
|total_employees|total_payroll|avg_salary|min_salary|max_salary|    stddev_salary|
+---------------+-------------+----------+----------+----------+-----------------+
|             20|      1684000|   84200.0|     68000|    105000|11794.73566781741|
+---------------+-------------+----------+----------+----------+-----------------+



In [402]:
# ── 10. GroupBy on multiple columns ──
print("Count and avg salary per department AND location:")
df.groupBy("department", "location").agg(
    count("*").alias("count"),
    avg("salary").alias("avg_salary")
).orderBy("department", "location").show()

Count and avg salary per department AND location:
+-----------+-------------+-----+-----------------+
| department|     location|count|       avg_salary|
+-----------+-------------+-----+-----------------+
|Engineering|San Francisco|    4|          99250.0|
|Engineering|      Seattle|    2|          95000.0|
|    Finance|     New York|    3|91333.33333333333|
|         HR|       Austin|    2|          71000.0|
|         HR|     New York|    1|          72000.0|
|  Marketing|       Boston|    2|          77500.0|
|  Marketing|  Los Angeles|    1|          81000.0|
|  Marketing|     New York|    1|          75000.0|
|      Sales|      Chicago|    3|75666.66666666667|
|      Sales|        Miami|    1|          71000.0|
+-----------+-------------+-----+-----------------+



In [403]:
# ── 11. Filtering after groupBy — equivalent to SQL HAVING ──
# Use .filter() on the result of .agg() to keep only certain groups
print("Departments with more than 3 employees AND avg salary > 80,000:")
df.groupBy("department").agg(
    count("*").alias("employee_count"),
    avg("salary").alias("avg_salary")
).filter(
    (col("employee_count") > 3) & (col("avg_salary") > 80000)
).orderBy("avg_salary", ascending=False).show()

Departments with more than 3 employees AND avg salary > 80,000:
+-----------+--------------+-----------------+
| department|employee_count|       avg_salary|
+-----------+--------------+-----------------+
|Engineering|             6|97833.33333333333|
+-----------+--------------+-----------------+



In [404]:
# ── 12. pivot() — cross-tabulation / pivot table ──
# Turns unique values of one column into separate output columns
print("Employee count per department × location (pivot table):")
df.groupBy("department").pivot("location").count().fillna(0).show()

Employee count per department × location (pivot table):
+-----------+------+------+-------+-----------+-----+--------+-------------+-------+
| department|Austin|Boston|Chicago|Los Angeles|Miami|New York|San Francisco|Seattle|
+-----------+------+------+-------+-----------+-----+--------+-------------+-------+
|      Sales|     0|     0|      3|          0|    1|       0|            0|      0|
|Engineering|     0|     0|      0|          0|    0|       0|            4|      2|
|         HR|     2|     0|      0|          0|    0|       1|            0|      0|
|    Finance|     0|     0|      0|          0|    0|       3|            0|      0|
|  Marketing|     0|     2|      0|          1|    0|       1|            0|      0|
+-----------+------+------+-------+-----------+-----+--------+-------------+-------+



### Quick Reference – Aggregation Cheatsheet

```python
from pyspark.sql.functions import (
    count, countDistinct,
    sum, sumDistinct,
    avg, mean,
    min, max,
    stddev, variance,
    first, last,
    collect_list, collect_set,
    percentile_approx,
    concat_ws
)

# Global aggregation (whole DataFrame)
df.agg(count("*"), avg("salary"), stddev("salary")).show()

# GroupBy + single aggregation
df.groupBy("dept").count().show()

# GroupBy + multiple aggregations using .agg()
df.groupBy("dept").agg(
    count("*").alias("n"),
    avg("salary").alias("avg_sal"),
    max("salary").alias("max_sal")
).show()

# Filter groups (SQL HAVING equivalent)
df.groupBy("dept").agg(count("*").alias("n")).filter(col("n") > 3).show()

# GroupBy multiple columns
df.groupBy("dept", "location").agg(avg("salary")).show()

# Pivot table
df.groupBy("dept").pivot("location").count().fillna(0).show()

# Collect all names in a group
df.groupBy("dept").agg(concat_ws(", ", collect_list("name")).alias("names")).show()
```

## 8. Sorting and Ordering Data

Use `orderBy()` or `sort()` to order data by one or more columns.

In [405]:
# Sort by salary (descending)
print("Top 5 highest paid employees:")
df.orderBy(col("salary").desc()).select("name", "department", "salary").show(5)

Top 5 highest paid employees:
+---------------+-----------+------+
|           name| department|salary|
+---------------+-----------+------+
|  Michael Brown|Engineering|105000|
|Jennifer Harris|Engineering|102000|
|    Joseph Hall|Engineering| 99000|
|   James Thomas|Engineering| 98000|
|     John Smith|Engineering| 95000|
+---------------+-----------+------+
only showing top 5 rows


In [406]:
# Sort by multiple columns
print("Employees sorted by department and salary:")
df.orderBy("department", col("salary").desc()).select("name", "department", "salary").show(10)

Employees sorted by department and salary:
+----------------+-----------+------+
|            name| department|salary|
+----------------+-----------+------+
|   Michael Brown|Engineering|105000|
| Jennifer Harris|Engineering|102000|
|     Joseph Hall|Engineering| 99000|
|    James Thomas|Engineering| 98000|
|      John Smith|Engineering| 95000|
|    David Wilson|Engineering| 88000|
| Daniel Robinson|    Finance| 94000|
|Patricia Jackson|    Finance| 91000|
|      Ryan Young|    Finance| 89000|
|      Karen King|         HR| 73000|
+----------------+-----------+------+
only showing top 10 rows


## 9. Joins Between DataFrames

Learn how to join two DataFrames using different join types (inner, left, right, outer).

In [407]:
# Create a second DataFrame - Department budgets
dept_data = [
    ("Engineering", 500000, "Tech"),
    ("Marketing", 250000, "Business"),
    ("Sales", 300000, "Business"),
    ("Finance", 200000, "Business"),
    ("HR", 150000, "Business"),
    ("IT", 180000, "Tech")  # IT department not in employees
]

dept_df = spark.createDataFrame(dept_data, ["department", "budget", "division"])
print("Department budgets:")
dept_df.show()

Department budgets:
+-----------+------+--------+
| department|budget|division|
+-----------+------+--------+
|Engineering|500000|    Tech|
|  Marketing|250000|Business|
|      Sales|300000|Business|
|    Finance|200000|Business|
|         HR|150000|Business|
|         IT|180000|    Tech|
+-----------+------+--------+



In [408]:
# Inner join - only matching records
print("Inner join:")
df.join(dept_df, "department", "inner").select("name", "department", "salary", "budget", "division").show(5)

Inner join:
+---------------+-----------+------+------+--------+
|           name| department|salary|budget|division|
+---------------+-----------+------+------+--------+
|    Joseph Hall|Engineering| 99000|500000|    Tech|
|Jennifer Harris|Engineering|102000|500000|    Tech|
|   James Thomas|Engineering| 98000|500000|    Tech|
|   David Wilson|Engineering| 88000|500000|    Tech|
|  Michael Brown|Engineering|105000|500000|    Tech|
+---------------+-----------+------+------+--------+
only showing top 5 rows


In [409]:
# Left join - all records from left (employees), matching from right
print("Left join:")
df.join(dept_df, "department", "left").select("name", "department", "salary", "budget").show(5)

Left join:
+----------------+-----------+------+------+
|            name| department|salary|budget|
+----------------+-----------+------+------+
|     Emily Davis|      Sales| 68000|300000|
|      John Smith|Engineering| 95000|500000|
|   Michael Brown|Engineering|105000|500000|
|    David Wilson|Engineering| 88000|500000|
|Jessica Martinez|         HR| 72000|150000|
+----------------+-----------+------+------+
only showing top 5 rows


## 10. Working with SQL Queries

Register DataFrames as temporary views and execute SQL queries using `spark.sql()`.

In [410]:
# Register DataFrame as temporary view
df.createOrReplaceTempView("employees")
dept_df.createOrReplaceTempView("departments")

print("Views created successfully!")

Views created successfully!


In [411]:
# Execute SQL query
sql_result = spark.sql("""
    SELECT department, COUNT(*) as employee_count, AVG(salary) as avg_salary
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
""")

print("SQL query result:")
sql_result.show()

SQL query result:
+-----------+--------------+-----------------+
| department|employee_count|       avg_salary|
+-----------+--------------+-----------------+
|Engineering|             6|97833.33333333333|
|    Finance|             3|91333.33333333333|
|  Marketing|             4|          77750.0|
|      Sales|             4|          74500.0|
|         HR|             3|71333.33333333333|
+-----------+--------------+-----------------+



In [412]:
# Complex SQL with JOIN
complex_sql = spark.sql("""
    SELECT e.name, e.salary, e.department, d.budget, d.division
    FROM employees e
    INNER JOIN departments d ON e.department = d.department
    WHERE e.salary > 80000
    ORDER BY e.salary DESC
""")

print("Complex SQL with join:")
complex_sql.show(5)

Complex SQL with join:
+---------------+------+-----------+------+--------+
|           name|salary| department|budget|division|
+---------------+------+-----------+------+--------+
|  Michael Brown|105000|Engineering|500000|    Tech|
|Jennifer Harris|102000|Engineering|500000|    Tech|
|    Joseph Hall| 99000|Engineering|500000|    Tech|
|   James Thomas| 98000|Engineering|500000|    Tech|
|     John Smith| 95000|Engineering|500000|    Tech|
+---------------+------+-----------+------+--------+
only showing top 5 rows


## 11. Handling Null Values

Learn how to identify and handle null values in your DataFrames.

In [413]:
# Check for null values in each column
from pyspark.sql.functions import col, isnan, when, count as spark_count

null_counts = df.select([spark_count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
print("Null counts per column:")
null_counts.show()

Null counts per column:
+-----------+----+----------+------+---+---------+--------+
|employee_id|name|department|salary|age|join_date|location|
+-----------+----+----------+------+---+---------+--------+
|          0|   0|         0|     0|  0|        0|       0|
+-----------+----+----------+------+---+---------+--------+



In [414]:
# Fill null values with defaults
# df_filled = df.fillna({"salary": 0, "location": "Unknown"})

# Drop rows with null values
# df_cleaned = df.dropna()

# Drop rows where specific columns have nulls
# df_cleaned = df.dropna(subset=["salary", "department"])

print("Null handling methods ready to use!")

Null handling methods ready to use!


## 12. Window Functions

Window functions allow calculations across rows related to the current row.

In [415]:
# Rank employees by salary within each department
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, dense_rank, row_number

windowSpec = Window.partitionBy("department").orderBy(col("salary").desc())

df_ranked = df.withColumn("rank", rank().over(windowSpec)) \
              .withColumn("dense_rank", dense_rank().over(windowSpec)) \
              .withColumn("row_number", row_number().over(windowSpec))

print("Employees ranked by salary within department:")
df_ranked.select("name", "department", "salary", "rank", "dense_rank", "row_number").orderBy("department", "rank").show(10)

Employees ranked by salary within department:
+----------------+-----------+------+----+----------+----------+
|            name| department|salary|rank|dense_rank|row_number|
+----------------+-----------+------+----+----------+----------+
|   Michael Brown|Engineering|105000|   1|         1|         1|
| Jennifer Harris|Engineering|102000|   2|         2|         2|
|     Joseph Hall|Engineering| 99000|   3|         3|         3|
|    James Thomas|Engineering| 98000|   4|         4|         4|
|      John Smith|Engineering| 95000|   5|         5|         5|
|    David Wilson|Engineering| 88000|   6|         6|         6|
| Daniel Robinson|    Finance| 94000|   1|         1|         1|
|Patricia Jackson|    Finance| 91000|   2|         2|         2|
|      Ryan Young|    Finance| 89000|   3|         3|         3|
|      Karen King|         HR| 73000|   1|         1|         1|
+----------------+-----------+------+----+----------+----------+
only showing top 10 rows


In [416]:
# Calculate running total and moving average
from pyspark.sql.functions import sum as spark_sum, avg as spark_avg

windowSpec2 = Window.partitionBy("department").orderBy("employee_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_window = df.withColumn("running_total_salary", spark_sum("salary").over(windowSpec2))

print("Running total of salaries by department:")
df_window.select("employee_id", "name", "department", "salary", "running_total_salary").orderBy("department", "employee_id").show(10)

Running total of salaries by department:
+-----------+----------------+-----------+------+--------------------+
|employee_id|            name| department|salary|running_total_salary|
+-----------+----------------+-----------+------+--------------------+
|        101|      John Smith|Engineering| 95000|               95000|
|        103|   Michael Brown|Engineering|105000|              200000|
|        105|    David Wilson|Engineering| 88000|              288000|
|        109|    James Thomas|Engineering| 98000|              386000|
|        112| Jennifer Harris|Engineering|102000|              488000|
|        117|     Joseph Hall|Engineering| 99000|              587000|
|        110|Patricia Jackson|    Finance| 91000|               91000|
|        115| Daniel Robinson|    Finance| 94000|              185000|
|        119|      Ryan Young|    Finance| 89000|              274000|
|        106|Jessica Martinez|         HR| 72000|               72000|
+-----------+----------------+------

## 13. Writing Data to Files

Learn how to write DataFrames to various file formats.

In [417]:
# Write to CSV
# df.write.mode("overwrite").csv("output/employees_csv", header=True)

# Write to Parquet (columnar format - more efficient)
# df.write.mode("overwrite").parquet("output/employees_parquet")

# Write to JSON
# df.write.mode("overwrite").json("output/employees_json")

# Write partitioned by a column
# df.write.mode("overwrite").partitionBy("department").parquet("output/employees_partitioned")

print("Data writing examples (commented out to avoid creating files)")

Data writing examples (commented out to avoid creating files)


## 14. String and Date Functions

Common string manipulations and date operations in PySpark.

In [418]:
# String operations
from pyspark.sql.functions import upper, lower, concat, lit, substring, length

df_strings = df.withColumn("name_upper", upper("name")) \
               .withColumn("name_lower", lower("name")) \
               .withColumn("name_length", length("name")) \
               .withColumn("first_name", substring("name", 1, 10))

print("String function examples:")
df_strings.select("name", "name_upper", "name_lower", "name_length").show(5)

String function examples:
+-------------+-------------+-------------+-----------+
|         name|   name_upper|   name_lower|name_length|
+-------------+-------------+-------------+-----------+
|   John Smith|   JOHN SMITH|   john smith|         10|
|Sarah Johnson|SARAH JOHNSON|sarah johnson|         13|
|Michael Brown|MICHAEL BROWN|michael brown|         13|
|  Emily Davis|  EMILY DAVIS|  emily davis|         11|
| David Wilson| DAVID WILSON| david wilson|         12|
+-------------+-------------+-------------+-----------+
only showing top 5 rows


In [419]:
# Date operations
from pyspark.sql.functions import to_date, year, month, datediff, current_date

df_dates = df.withColumn("join_date_formatted", to_date("join_date", "yyyy-MM-dd")) \
             .withColumn("join_year", year("join_date")) \
             .withColumn("join_month", month("join_date")) \
             .withColumn("days_employed", datediff(current_date(), "join_date"))

print("Date function examples:")
df_dates.select("name", "join_date", "join_year", "join_month", "days_employed").show(5)

Date function examples:
+-------------+----------+---------+----------+-------------+
|         name| join_date|join_year|join_month|days_employed|
+-------------+----------+---------+----------+-------------+
|   John Smith|2019-03-15|     2019|         3|         2534|
|Sarah Johnson|2020-06-01|     2020|         6|         2090|
|Michael Brown|2018-01-20|     2018|         1|         2953|
|  Emily Davis|2021-02-10|     2021|         2|         1836|
| David Wilson|2019-11-05|     2019|        11|         2299|
+-------------+----------+---------+----------+-------------+
only showing top 5 rows


## 15. Advanced Operations and Best Practices

Key tips and advanced techniques for working with PySpark.

### PySpark Best Practices:

1. **Use `select()` early**: Select only the columns you need to reduce data transfer
2. **Filter early**: Apply filters as early as possible in your pipeline
3. **Avoid `collect()`**: Don't collect large DataFrames to the driver; use actions like `show()`, `count()`, or write to files
4. **Cache wisely**: Use `.cache()` or `.persist()` for DataFrames used multiple times
5. **Partition properly**: Choose the right number of partitions for your data size
6. **Use built-in functions**: PySpark's built-in functions are optimized; avoid UDFs when possible
7. **Broadcast small DataFrames**: Use `broadcast()` for joins with small tables
8. **Monitor your jobs**: Use Spark UI to understand and optimize your jobs
9. **Choose the right file format**: Parquet is usually better than CSV for big data
10. **Clean up resources**: Stop SparkSession when done

In [420]:
# Cache a DataFrame for reuse
df.cache()
print(f"DataFrame cached. Row count: {df.count()}")

# Uncache when done
# df.unpersist()

DataFrame cached. Row count: 20


26/02/20 20:56:46 WARN CacheManager: Asked to cache already cached data.


In [421]:
# Get distinct values
print("Unique departments:")
df.select("department").distinct().show()

# Get distinct count
print(f"Number of unique departments: {df.select('department').distinct().count()}")

Unique departments:
+-----------+
| department|
+-----------+
|      Sales|
|Engineering|
|         HR|
|    Finance|
|  Marketing|
+-----------+

Number of unique departments: 5


In [422]:
# Remove duplicate rows
# df_no_duplicates = df.dropDuplicates()

# Remove duplicates based on specific columns
# df_unique = df.dropDuplicates(["employee_id"])

print("Duplicate removal methods available")

Duplicate removal methods available


## 16. Conclusion and Next Steps

Congratulations! You've learned the fundamentals of PySpark including:
- DataFrame creation and basic operations
- Data filtering and selection
- Aggregations and grouping
- Joins between DataFrames
- SQL queries
- Window functions
- String and date operations
- Writing data to various formats

### Next Steps:
1. Practice with larger datasets
2. Learn about Spark MLlib for machine learning
3. Explore Spark Streaming for real-time data processing
4. Study performance optimization techniques
5. Work with cloud platforms (AWS EMR, Azure HDInsight, Databricks)

### Useful Resources:
- [Official PySpark Documentation](https://spark.apache.org/docs/latest/api/python/)
- [Spark SQL Guide](https://spark.apache.org/docs/latest/sql-programming-guide.html)
- [PySpark API Reference](https://spark.apache.org/docs/latest/api/python/reference/index.html)

In [423]:
# Stop the SparkSession when done
# spark.stop()
print("Remember to stop your SparkSession when finished!")

Remember to stop your SparkSession when finished!
